[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C77_Video_World_Models_Course/03_temporal_drift/03_temporal_drift.ipynb)

# C77 模块 03 · 时序一致性与自回归漂移

四件事：

1. **漂移不是必然的**：误差按 $|\hat a|^k$ 演化，稳定动力学会**遗忘**错误；
2. **一步拟合系统性低估持续性** $\Rightarrow$ rollout 方差只有真值的 35%–49%；
3. **少数模型会爆炸**（$a{=}0.995, n{=}30$ 时 28.6%），而**均值指标同时错判两支**；
4. **分布的错误不会自我纠正**（噪声尺度错 $\Rightarrow$ 不动点错）。

纯 numpy / CPU / 离线，不训练任何网络。

In [ ]:
import numpy as np

def ar1(a, n, seed, sig=1.0):
    '''AR(1) 序列，从稳态分布起始。'''
    r = np.random.default_rng(seed)
    x = np.zeros(n)
    x[0] = r.normal(0, sig / np.sqrt(max(1 - a*a, 1e-6)))
    for t in range(1, n):
        x[t] = a * x[t-1] + r.normal(0, sig)
    return x

def fit_a(x):
    '''一步最小二乘（= 教师强制训练的解析解）。'''
    return float(np.sum(x[:-1] * x[1:]) / np.sum(x[:-1]**2))

print('本模块用 AR(1) 当「视频动力学」的最小模型：')
print('  · 一步最小二乘 = 教师强制训练的解析最优解')
print('  · 自回归 rollout = 反复代入自己的输出')
print('两者都不需要神经网络就能复现「生成的视频越往后越静止」这个现象。')

## 1. 误差增长律：$|\hat a|^k$

初始帧带误差 $\delta$，$k$ 步后是 $|\hat a|^k \delta$。

In [ ]:
print('   â      1 步       10 步        50 步       200 步      增长律')
for ah in (0.50, 0.90, 0.99, 1.00, 1.01, 1.05):
    row = [ah**k for k in (1, 10, 50, 200)]
    law = '收敛到 0' if ah < 1 else ('恒定' if ah == 1 else '指数爆炸')
    print(f'  {ah:5.2f}  ' + '  '.join(f'{v:10.4g}' for v in row) + f'   {law}')

# 数值验证：真的按几何级数走
def rollout_error(ah, delta, k):
    e = delta
    for _ in range(k):
        e = ah * e
    return abs(e)
for ah in (0.5, 0.9, 0.99, 1.01):
    for k in (10, 50):
        assert abs(rollout_error(ah, 1.0, k) - ah**k) < 1e-12 * max(1, ah**k)
print()
print('✅ 误差严格按几何级数 |â|^k 演化（这是恒等式，不是近似）')
print()
print('-> |â| < 1 时**任何单帧的错误都会被指数地遗忘**。')
print('   所以「一帧生成得不好，后面就全毁了」在稳定动力学下不成立。')
print('   而现实的视频动力学在多数尺度上是稳定的（物体会停、光照回均值、镜头稳住）。')
print()
print('   那长视频为什么确实会漂？因为真正累积的不是单条轨迹的误差，')
print('   而是**分布的错误** —— 见第 2、4 节。')

## 2. 一步拟合对持续性的估计向下有偏

$E[\hat a] - a \approx -(1+3a)/n$（Kendall 1954 / Marriott–Pope 1954）。

In [ ]:
print('  真 a    n      â 中位数    â 均值     偏差(均值)   理论 -(1+3a)/n   实测/理论')
bias_tab = {}
for a in (0.90, 0.98):
    for n in (50, 200, 1000):
        ahs = np.array([fit_a(ar1(a, n, s)) for s in range(6000)])
        bias = ahs.mean() - a
        theo = -(1 + 3*a) / n
        bias_tab[(a, n)] = (float(np.median(ahs)), float(ahs.mean()), float(bias))
        print(f'  {a:5.2f}  {n:5d}   {np.median(ahs):+.5f}  {ahs.mean():+.5f}  '
              f'{bias:+.5f}     {theo:+.5f}      {bias/theo:8.3f}')
    print()

# 偏差为负、且按 1/n 缩小
for a in (0.90, 0.98):
    for n in (50, 200, 1000):
        assert bias_tab[(a, n)][2] < 0, f'a={a}, n={n}: 偏差应为负'
    b50, b1000 = bias_tab[(a, 50)][2], bias_tab[(a, 1000)][2]
    ratio = b50 / b1000
    print(f'  a={a}: 偏差从 n=50 的 {b50:+.5f} 缩到 n=1000 的 {b1000:+.5f}'
          f'（缩小 {ratio:.1f} 倍，n 比是 20 倍）')
    assert 8 < ratio < 40, f'偏差应大致按 1/n 缩小，实测 {ratio:.1f} 倍'

print()
print('✅ 偏差恒为负、且按 ~1/n 缩小 —— 符号与标度都与理论一致')
print('✅ 实测偏差约是理论式的一半（理论式给的是量级与符号）')

In [ ]:
# 后果：rollout 的稳态方差偏小 —— 生成的视频变静止
def rollout_var(a, n, seeds=400, horizon=400, burn=200):
    '''在长度 n 的序列上拟合，然后 rollout，返回稳态方差的中位数与均值。'''
    vs = []
    for s in range(seeds):
        x = ar1(a, n, 10_000 + s)
        aa = fit_a(x)
        sg = (x[1:] - aa * x[:-1]).std()          # 用训练残差当噪声尺度
        y = np.zeros(horizon)
        y[0] = x[-1]
        rr = np.random.default_rng(50_000 + s)
        for t in range(1, horizon):
            y[t] = aa * y[t-1] + rr.normal(0, sg)
        vs.append(y[burn:].var())
    vs = np.array(vs)
    return float(np.median(vs)), float(vs.mean())

print('  真 a    n     真稳态方差   rollout 方差(中位数)   比值     rollout 方差(均值)')
var_tab = {}
for a in (0.90, 0.98):
    for n in (50, 200, 1000):
        med, mean = rollout_var(a, n)
        vt = 1.0 / (1 - a*a)
        var_tab[(a, n)] = (med, mean, vt)
        print(f'  {a:5.2f}  {n:5d}  {vt:11.3f}   {med:19.3f}   {med/vt:.4f}   {mean:.3e}')
    print()

# 方差系统性偏小
for key, (med, mean, vt) in var_tab.items():
    assert med < vt, f'{key}: rollout 方差应偏小'
assert var_tab[(0.98, 50)][0] / var_tab[(0.98, 50)][2] < 0.5, \
    'a=0.98, n=50 时方差比应 <0.5'
# n 越大越接近真值
for a in (0.90, 0.98):
    r50 = var_tab[(a, 50)][0] / var_tab[(a, 50)][2]
    r1000 = var_tab[(a, 1000)][0] / var_tab[(a, 1000)][2]
    assert r1000 > r50, f'a={a}: n 越大方差比应越接近 1'

print(f'✅ a=0.98 时 rollout 的运动能量只有真实的 '
      f'{var_tab[(0.98,50)][0]/var_tab[(0.98,50)][2]*100:.0f}%–'
      f'{var_tab[(0.98,1000)][0]/var_tab[(0.98,1000)][2]*100:.0f}%')
print()
print('-> 「越往后越糊、越静止」有一个不需要神经网络的解释:')
print('   即使模型类完全正确、即使训练完美收敛到该目标的最优解，')
print('   **一步拟合本身就会低估动力学的持续性** —— 这是目标函数问题，不是能力问题。')
print('   这也解释了为什么「多步/rollout 损失」「scheduled sampling」有用：它们改的正是这个目标。')

## 3. 尾部：少数模型会爆炸，而均值指标同时错判两支

In [ ]:
print('  真 a     n      â 中位数   â 均值    P(|â|>1)   P(|â|>1.01)')
tail = {}
for a in (0.900, 0.980, 0.995):
    for n in (30, 50, 100, 200, 1000):
        ahs = np.array([fit_a(ar1(a, n, s)) for s in range(6000)])
        p1 = float(np.mean(np.abs(ahs) > 1))
        tail[(a, n)] = (float(np.median(ahs)), p1)
        print(f'  {a:5.3f}  {n:5d}   {np.median(ahs):+.5f}  {ahs.mean():+.5f}   '
              f'{p1*100:6.2f}%    {np.mean(np.abs(ahs)>1.01)*100:6.2f}%')
    print()

# a 越接近 1、n 越小，不稳定拟合越多
assert tail[(0.995, 30)][1] > 0.2, f'a=0.995, n=30 时应有 >20% 不稳定'
assert tail[(0.900, 100)][1] < 0.01, f'a=0.900, n=100 时应几乎没有不稳定'
for a in (0.980, 0.995):
    seq = [tail[(a, n)][1] for n in (30, 50, 100, 200, 1000)]
    for i in range(1, len(seq)):
        assert seq[i] <= seq[i-1] + 1e-9, f'a={a}: 不稳定比例应随 n 下降 {seq}'

print(f'✅ a=0.995, n=30 时 {tail[(0.995,30)][1]*100:.1f}% 的拟合给出 |â|>1（会指数爆炸）')
print(f'✅ a=0.900, n=100 时降到 {tail[(0.900,100)][1]*100:.2f}%')

print()
print('两支的症状完全相反，而**均值指标同时错判两者**:')
print('  真 a    n     真稳态方差   方差中位数     方差均值        均值/中位数')
for a, n in [(0.90, 50), (0.98, 50), (0.98, 200), (0.98, 1000)]:
    med, mean, vt = var_tab[(a, n)]
    print(f'  {a:5.2f}  {n:5d}  {vt:11.3f}   {med:11.3f}   {mean:.3e}   {mean/med:.3e}')

_med, _mean, _ = var_tab[(0.98, 50)]
assert _mean / _med > 1e10, f'a=0.98, n=50 时均值应被爆炸样本主导，实测 {_mean/_med:.2e}'
print()
print(f'✅ a=0.98, n=50: 中位数 {_med:.2f}，均值 {_mean:.3e} —— 差 '
      f'{np.log10(_mean/_med):.0f} 个数量级')
print(f'   {tail[(0.98,50)][1]*100:.0f}% 的爆炸样本完全主导了均值。')
print()
print('-> 所以长视频指标必须报**中位数 + 崩坏率**，而不是平均。')
print('   在一批生成的长视频上报平均任何指标（FVD、逐帧 PSNR、运动幅度）')
print('   都会被少数崩坏样本主导，而那些样本本该被单独拎出来看。')

## 4. 分布的错误不会自我纠正

真 $a{=}0.95$、真噪声 $\text{sd}{=}1.0$，真稳态 $\text{sd} = 1/\sqrt{1-a^2}$。
如果模型的噪声尺度错了会怎样。

In [ ]:
a = 0.95
true_sd = 1.0 / np.sqrt(1 - a*a)
print(f'真 a={a}，真噪声 sd=1.0，真稳态 sd = {true_sd:.4f}')
print()
print('  模型噪声 sd   第 5 步 sd   第 50 步   第 500 步   稳态 sd / 真值')
fix = {}
for sg in (0.50, 0.80, 1.00, 1.20, 2.00):
    r = np.random.default_rng(0)
    N = 4000
    y = np.zeros((N, 501))
    y[:, 0] = r.normal(0, true_sd, N)              # 从**正确的**稳态起始
    for t in range(1, 501):
        y[:, t] = a * y[:, t-1] + r.normal(0, sg, N)
    sds = [float(y[:, k].std()) for k in (5, 50, 500)]
    fix[sg] = sds[-1] / true_sd
    print(f'  {sg:11.2f}   {sds[0]:10.3f}   {sds[1]:8.3f}   {sds[2]:9.3f}   '
          f'{sds[-1]/true_sd:14.4f}')
    # 稳态 sd 与噪声 sd 成正比
    assert abs(sds[-1]/true_sd - sg) < 0.02, \
        f'稳态比值应等于噪声比值 {sg}，得到 {sds[-1]/true_sd:.4f}'

print()
print('✅ 稳态 sd 的比值恰好等于噪声 sd 的比值 —— 因为稳态 sd = σ/sqrt(1−a²)')
print('✅ 即使从**正确的**稳态分布起始，错的噪声尺度也会把它拉到错的不动点')
print()
print('-> 单条轨迹的误差可以被遗忘（第 1 节），但**分布的错误是持久的**。')
print('   收缩性保证「初值不重要」，它**不**保证「收敛到正确的分布」——')
print('   两件事经常被混为一谈。')
print()
print('   实践含义：采样温度 / 引导强度在自回归 rollout 里直接决定不动点的方差。')
print('   CFG 偏高 -> 越来越「锐利到失真」；偏低 -> 越来越糊。')
print('   这两个方向都**不是**误差累积，而是不动点选错了 —— 修法也不同：')
print('   误差累积要改训练目标（第 2 节），不动点错要改采样参数。')

## ✏️ 练习 1：多步损失到底改了什么

「用 rollout 损失修漂移」是一条流行的建议。这个练习把它测清楚。

实现 `fit_a_multistep(x, k)`：最小化 $k$ 步 rollout 的总误差
$\sum_t \sum_{j=1}^{k} (a^j x_t - x_{t+j})^2$（对标量 $a$ 用网格搜索即可）。

自测会验证三件事，其中第一件与直觉相反。

In [ ]:
def fit_a_multistep(x, k, grid=None):
    '''最小化 k 步 rollout 总误差的 a。

    目标: sum_t sum_{j=1..k} (a^j * x_t − x_{t+j})^2

    参数
    ----
    x    : 训练序列
    k    : rollout 步数（k=1 时应与一步最小二乘接近）
    grid : 候选 a（默认 −1.2 到 1.2 的 2401 个点）

    返回
    ----
    float : 最优 a
    '''
    if grid is None:
        grid = np.linspace(-1.2, 1.2, 2401)
    # TODO: 对每个候选 a 算目标函数，返回 argmin 对应的 a
    raise NotImplementedError

In [ ]:
# 自测
print('  (a) 多步损失**不**修 â 的向下偏差（与直觉相反）')
print('    真 a    n     一步中位   k=2      k=4      k=8      k=16     偏差最小的 k')
_bestk = []
for _a in (0.90, 0.98):
    for _n in (50, 200, 1000):
        _m1 = float(np.median([fit_a(ar1(_a, _n, s)) for s in range(200)]))
        _ms = {_k: float(np.median([fit_a_multistep(ar1(_a, _n, s), _k)
                                    for s in range(200)])) for _k in (2, 4, 8, 16)}
        _b = {1: abs(_m1 - _a)}
        _b.update({_k: abs(v - _a) for _k, v in _ms.items()})
        _bk = min(_b, key=_b.get)
        _bestk.append(_bk)
        print(f'    {_a:5.2f} {_n:5d}  {_m1:+.5f}  ' +
              '  '.join(f'{_ms[_k]:+.5f}' for _k in (2,4,8,16)) + f'   k={_bk}')
        # 全部估计仍然向下偏
        assert _m1 < _a and all(v < _a + 1e-9 for v in _ms.values()), \
            '一步与多步的估计都应向下偏'
assert len(set(_bestk)) > 1, f'最优 k 应在不同配置下变化（说明它不是系统性的）：{_bestk}'
print(f'    -> 偏差最小的 k 在 {sorted(set(_bestk))} 之间跳，**不是系统性的**。')
print('       原因：对**正确指定**的 AR(1)，一步最小二乘已经接近有效估计，')
print('       多步损失没有额外信息可用。')

print()
print('  (b) 模型正确指定时，多步损失几乎什么也不改')
def _hpred_err(a, seqs, H):
    '''用 AR(1) 系数 a 做 H 步预测的相对误差。'''
    num = den = 0.0
    for x in seqs:
        n = len(x)
        pred, targ = (a**H) * x[:n-H], x[H:]
        num += float(((pred - targ)**2).sum())
        den += float(((targ - targ.mean())**2).sum())
    return float(np.sqrt(num / den))

_tr = [ar1(0.95, 200, s) for s in range(60)]
_te = [ar1(0.95, 400, 10_000 + s) for s in range(60)]
print('    拟合用的 k   â        H=1 误差   H=4       H=8       H=16')
_e_ok = {}
for _k in (1, 2, 4, 8, 16):
    _ah = (float(np.median([fit_a(x) for x in _tr])) if _k == 1
           else float(np.median([fit_a_multistep(x, _k) for x in _tr])))
    _row = {_H: _hpred_err(_ah, _te, _H) for _H in (1, 4, 8, 16)}
    _e_ok[_k] = _row
    print(f'    {_k:10d}   {_ah:+.4f}   ' + '  '.join(f'{_row[_H]:.5f}' for _H in (1,4,8,16)))
for _H in (1, 4, 8, 16):
    _spread = max(_e_ok[_k][_H] for _k in _e_ok) - min(_e_ok[_k][_H] for _k in _e_ok)
    assert _spread < 0.005, f'H={_H}: 各 k 的误差差异应 <0.005，实测 {_spread:.5f}'
print('    -> 各 k 的误差只差在第 4 位小数。多步损失在这里是白做的。')

print()
print('  (c) 模型**误配**时才是真实的取舍')
def _ar2(n, seed, rad=0.95, per=6.0, sig=0.3):
    '''振荡的 AR(2) —— AR(1) 模型对它是误配的。'''
    r = np.random.default_rng(seed)
    w = 2*np.pi/per
    c1, c2 = 2*rad*np.cos(w), -rad**2
    x = np.zeros(n)
    x[0], x[1] = r.normal(), r.normal()
    for t in range(2, n):
        x[t] = c1*x[t-1] + c2*x[t-2] + r.normal(0, sig)
    return x

_tr2 = [_ar2(200, s) for s in range(60)]
_te2 = [_ar2(400, 10_000 + s) for s in range(60)]
print('    拟合用的 k   â        H=1 误差   H=4       H=8       H=16')
_e_mis = {}
for _k in (1, 2, 4, 8, 16):
    _ah = (float(np.median([fit_a(x) for x in _tr2])) if _k == 1
           else float(np.median([fit_a_multistep(x, _k) for x in _tr2])))
    _row = {_H: _hpred_err(_ah, _te2, _H) for _H in (1, 4, 8, 16)}
    _e_mis[_k] = _row
    print(f'    {_k:10d}   {_ah:+.4f}   ' + '  '.join(f'{_row[_H]:.5f}' for _H in (1,4,8,16)))

# k=1 在 H=1 最好
assert _e_mis[1][1] == min(_e_mis[_k][1] for _k in _e_mis), 'k=1 应在 H=1 最好'
# 但 k=1 在 H=4 时**比预测均值还差**（误差 > 1.0）
assert _e_mis[1][4] > 1.0, f'k=1 在 H=4 应比均值预测更差，实测 {_e_mis[1][4]:.5f}'
# 而 k>=2 避免了这一点
assert _e_mis[4][4] < 1.005, f'k=4 在 H=4 应不比均值预测差，实测 {_e_mis[4][4]:.5f}'

print()
print(f'✅ (a) 多步损失**不**修 â 的向下偏差 —— 最优 k 在 {sorted(set(_bestk))} 之间跳')
print(f'✅ (b) 模型正确指定时它几乎什么也不改（各 k 差在第 4 位小数）')
print(f'✅ (c) 模型误配时是真实取舍：k=1 在 H=1 最好（{_e_mis[1][1]:.5f}），')
print(f'      但在 H=4 时是 {_e_mis[1][4]:.5f} > 1.0 —— **比直接预测均值还差**；')
print(f'      而 k=4 在 H=4 是 {_e_mis[4][4]:.5f}，避免了这一点。')
print()
print('   所以「用 rollout 损失修漂移」这条建议需要被限定:')
print('     · 它修不了第 2 节那个目标函数偏差（那是有限样本效应，不是目标形状问题）；')
print('     · 它在模型正确指定时是白做的；')
print('     · 它的真实作用是在**误配**下把精度从短跨度挪到长跨度 ——')
print('       而「一步拟合的模型在长跨度上可能比预测均值还差」是这个挪动的理由。')

## ✏️ 练习 2：崩坏率是一个可测量的指标

正文说长视频指标应报「中位数 + 崩坏率」。
实现 `collapse_rate(a, n, seeds, horizon, thresh)`：返回
rollout 过程中<strong>任一帧</strong>的能量超过训练分布 `thresh` 倍的比例。

In [ ]:
def collapse_rate(a, n, seeds=400, horizon=200, thresh=10.0):
    '''崩坏率：rollout 中任一帧的 |x| 超过训练分布 sd 的 thresh 倍的样本比例。

    参数
    ----
    a, n     : 真动力学与训练序列长度
    seeds    : 重复次数
    horizon  : rollout 长度
    thresh   : 判定倍数（相对训练序列的 sd）

    返回
    ----
    (rate, median_var) : 崩坏率，以及**未崩坏样本**的 rollout 方差中位数
    '''
    # TODO: 对每个 seed：x = ar1(a, n, 10_000+s)；aa = fit_a(x)；
    #       sg = 训练残差 sd；从 x[-1] 起 rollout horizon 步（噪声 sd = sg）；
    #       若 max|y| > thresh * x.std() 记为崩坏；
    #       返回 (崩坏比例, 未崩坏样本的 y[horizon//2:].var() 的中位数)
    raise NotImplementedError

In [ ]:
# 自测
print('  真 a     n     崩坏率    未崩坏样本的方差中位数   真稳态方差   比值')
rates = {}
for _a in (0.90, 0.98, 0.995):
    for _n in (30, 100, 1000):
        _rate, _med = collapse_rate(_a, _n)
        _vt = 1.0 / (1 - _a*_a)
        rates[(_a, _n)] = _rate
        print(f'  {_a:5.3f}  {_n:5d}   {_rate*100:6.2f}%   {_med:20.3f}   '
              f'{_vt:10.3f}   {_med/_vt:.4f}')
    print()

# 崩坏率随 n 下降、随 a→1 上升
for _a in (0.98, 0.995):
    _seq = [rates[(_a, _n)] for _n in (30, 100, 1000)]
    for _i in range(1, len(_seq)):
        assert _seq[_i] <= _seq[_i-1] + 1e-9, f'a={_a}: 崩坏率应随 n 下降 {_seq}'
assert rates[(0.995, 30)] > rates[(0.90, 30)], 'a 越接近 1 崩坏率应越高'
assert rates[(0.90, 1000)] < 0.02, 'a=0.90, n=1000 时崩坏率应很低'

# 未崩坏样本的方差仍系统性偏小 —— 两支是独立的问题
_r, _m = collapse_rate(0.98, 100)
assert _m / (1/(1-0.98**2)) < 0.7, \
    '未崩坏样本的方差仍应明显偏小（这是第 2 节的偏差，与崩坏无关）'

print(f'✅ 崩坏率随 n 下降（a=0.995: {rates[(0.995,30)]*100:.1f}% -> '
      f'{rates[(0.995,1000)]*100:.1f}%）、随 a→1 上升')
print(f'✅ 而**未崩坏样本**的方差比仍只有 {_m/(1/(1-0.98**2)):.3f} ——')
print('   两支是独立的问题：排除崩坏样本后，剩下的仍然过度衰减。')
print()
print('   -> 所以「中位数 + 崩坏率」这两个数缺一不可：')
print('      崩坏率捕捉尾部那一支，中位数捕捉主体那一支。')
print('      只报平均值会把两者混成一个无意义的数。')

## ✏️ 练习 3：锚帧间隔的账

实现 `anchor_error(ah, T, m)`：用间隔 $m$ 的锚帧重置时，
第 $T$ 帧的误差放大倍数。

- 无锚帧（$m \geq T$）：误差 $= |\hat a|^T$
- 有锚帧：每 $m$ 步重置，所以最大连续 rollout 长度是 $m-1$，
  而误差在锚帧处被清零 $\Rightarrow$ 最终误差 $= |\hat a|^{(T-1) \bmod m}$ 级别。

用它算出「要把误差压到 $\varepsilon$ 以下所需的锚帧间隔」。

In [ ]:
def anchor_error(ah, T, m):
    '''间隔 m 的锚帧下，第 T 帧的误差放大倍数（初始误差 = 1）。

    锚帧在 t = 0, m, 2m, ... 处把误差清零（那些帧是给定的/另外生成的）。
    所以第 T 帧的误差 = |ah|^(距离最近的前一个锚帧的步数)。

    参数
    ----
    ah : 学到的谱半径 |â|
    T  : 目标帧序号
    m  : 锚帧间隔（m >= T 表示只有 t=0 一个锚帧）

    返回
    ----
    float : |ah|^k，k = T - (最大的 <= T 的锚帧位置)
    '''
    # TODO: 找到 <= T 的最大锚帧位置 (T // m) * m；k = T - 那个位置；返回 ah**k
    raise NotImplementedError

In [ ]:
# 自测
print('  (a) 稳定动力学（â=0.95）。注意「无锚」要用 m > T ——')
print('      若取 m = T，锚帧恰好落在第 T 帧上，误差会被算成 1（这是一个容易踩的坑）')
print('     T    无锚(m>T)     m=16      m=8       m=4     m=T（错的写法）')
for _T in (16, 64, 256):
    _row = [anchor_error(0.95, _T, _T + 1)] +            [anchor_error(0.95, _T, _m) for _m in (16, 8, 4)] +            [anchor_error(0.95, _T, _T)]
    print(f'   {_T:4d}   ' + '  '.join(f'{v:9.3e}' for v in _row))
    assert abs(_row[0] - 0.95**_T) < 1e-12, '无锚应等于 |â|^T'
    assert abs(_row[-1] - 1.0) < 1e-12, 'm=T 时锚帧落在第 T 帧上，误差恰为 1'

print()
print('  (b) **不稳定**动力学（â=1.02）—— 锚帧的收益是指数级的:')
print('     T    无锚(m>T)      m=16       m=8        m=4     无锚/m=8')
for _T in (16, 64, 256, 1024):
    _no = anchor_error(1.02, _T, _T + 1)
    _row = [_no] + [anchor_error(1.02, _T, _m) for _m in (16, 8, 4)]
    print(f'   {_T:4d}   ' + '  '.join(f'{v:10.3e}' for v in _row) +
          f'   {_no/_row[2]:.3e}')
    # 改善倍数 = |â|^T / |â|^(T mod m)，T 是 m 的倍数时就是 |â|^T
    assert _no / _row[2] > 1.0, f'T={_T}: 锚帧应带来改善'
    if _T >= 256:
        assert _no / _row[2] > 1e2,             f'T={_T}: 改善应 >100 倍（1.02^{_T} = {1.02**_T:.3g}），实测 {_no/_row[2]:.3g}'

# 锚帧位置的正确性
assert abs(anchor_error(0.9, 16, 17) - 0.9**16) < 1e-12   # m>T：只有 t=0 一个锚帧
assert abs(anchor_error(0.9, 16, 16) - 0.9**0) < 1e-12    # m=T：锚帧正落在第 16 帧上
assert abs(anchor_error(0.9, 16, 8) - 0.9**0) < 1e-12     # t=16 也正好是锚帧
assert abs(anchor_error(0.9, 17, 8) - 0.9**1) < 1e-12     # t=16 是锚帧，差 1 步
assert abs(anchor_error(0.9, 23, 8) - 0.9**7) < 1e-12     # t=16 是锚帧，差 7 步

# 求所需锚帧间隔
print()
print('  (c) 锚帧只在 |â| > 1 时有用 —— 而这正是第 1 节的结论')
print('      |â| < 1 时误差自己就衰减，chunk 内的最坏值恰好在锚帧上（= 1），')
print('      所以加锚帧**一点也不减小**最坏误差。')
print()
print('     â      chunk 内最坏放大倍数 = |â|^(m-1)')
print('              m=2      m=4      m=8      m=16     m=64')
for _ah in (0.90, 0.99, 1.00, 1.02, 1.05):
    _row = [_ah**(_m-1) for _m in (2, 4, 8, 16, 64)]
    print(f'    {_ah:5.2f}   ' + '  '.join(f'{v:7.3g}' for v in _row))
    if _ah < 1.0:
        # 稳定：最坏值恒为 1（在锚帧处），与 m 无关
        _worst = max(anchor_error(_ah, _t, 8) for _t in range(240, 257))
        assert abs(_worst - 1.0) < 1e-12,             f'â={_ah}: chunk 内最坏值应恒为 1（在锚帧上），得到 {_worst}'
print()
print('     -> |â|<1 那两行的最坏放大倍数都 <= 1（误差在衰减）；')
print('        |â|>1 那两行 > 1，而 m 越小越接近 1。')
print()
print('  (d) 所以正确的问法是：|â| > 1 时，多密的锚帧能把放大倍数压到 F 以下')
print('      闭式解: |â|^(m-1) <= F  <=>  m <= 1 + ln F / ln|â|')
print('     â      F=1.5 所需 m     F=2 所需 m     F=10 所需 m')
for _ah in (1.01, 1.02, 1.05, 1.10):
    _row = []
    for _F in (1.5, 2.0, 10.0):
        _m_max = int(np.floor(1 + np.log(_F)/np.log(_ah)))
        # 验证闭式
        assert _ah**(_m_max - 1) <= _F + 1e-9, f'â={_ah}, F={_F}: 闭式应成立'
        assert _ah**_m_max > _F, f'â={_ah}, F={_F}: m+1 应超出'
        _row.append(_m_max)
    print(f'    {_ah:5.2f}   {_row[0]:12d}   {_row[1]:12d}   {_row[2]:13d}')
print()
print('     -> 例如 â=1.02、要求放大不超过 2 倍：m <= 1 + ln2/ln1.02 = '
      f'{int(np.floor(1+np.log(2)/np.log(1.02)))}，即每 '
      f'{int(np.floor(1+np.log(2)/np.log(1.02)))} 帧要一个锚帧。')
print('        而不加锚帧时 T=256 的放大是 '
      f'{1.02**256:.3g} 倍。')

print()
print('✅ 锚帧把有效 rollout 长度从 T 压到 <= m，误差从 |â|^T 变成 |â|^(<=m)')
print(f'✅ 在不稳定动力学下改善是**指数级**的：â=1.02 时')
print(f'   T=64 改善 {1.02**64:.2f} 倍，T=256 是 {1.02**256:.3g} 倍，T=1024 是 {1.02**1024:.3g} 倍')
print('   （改善倍数恰为 |â|^T，因为 T 是 m 的倍数时锚帧把误差清零）')
print()
print('   -> 这就是层次化生成的账：一次性生成一小段（段内无漂移），')
print('      再用锚帧把小段拼长（段间漂移 = 段数）。')
print('      代价见模块 00：段长 m 的一次性生成代价 ∝ m²。')

## ✏️ 练习 4：区分「误差累积」与「不动点错」

给一批生成的序列，怎么判断它属于哪一支？关键区别是**序列是否平稳**：

- **不动点错** → 序列已经平稳（前后半段方差一致），只是水平不对；
- **误差还在累积** → 序列不平稳（后半段方差明显不同）。

实现 `diagnose_drift(gen_seqs, ref_seqs, tol=0.20)`。

两个设计要点，自测会分别验证：

1. 它必须接收**一批**序列而不是一条（(a) 部分）；
2. 能量比必须相对**一批真实序列**算，而不是相对理论值——
   因为 sd 估计量本身在短序列上向下有偏（(b) 部分）。

In [ ]:
def diagnose_drift(gen_seqs, ref_seqs, tol=0.20):
    '''区分「不动点错」与「误差还在累积」。

    参数
    ----
    gen_seqs : 生成的序列列表
    ref_seqs : **真实**序列列表（同样长度、同样统计量口径）
    tol      : 判定「平稳」与「能量正常」的相对容差

    返回
    ----
    dict : {'energy_ratio', 'stat_ratio', 'is_stationary', 'verdict'}
      energy_ratio = median(gen 后半段 sd) / median(ref 后半段 sd)
      stat_ratio   = median(gen 后半段 sd / 前半段 sd)
      is_stationary = |stat_ratio − 1| < tol
      verdict: 'ok' | 'fixed_point' | 'compounding'
    '''
    # TODO: 对 gen_seqs 算 (后半 sd / 前半 sd) 的中位数 = stat_ratio；
    #       energy_ratio = median(gen 后半 sd) / median(ref 后半 sd)；
    #       不平稳 -> 'compounding'；平稳且 |energy_ratio−1|<tol -> 'ok'；
    #       平稳但能量偏离 -> 'fixed_point'
    raise NotImplementedError

In [ ]:
# 自测
_a, _true_sd = 0.95, 1.0/np.sqrt(1-0.95**2)
def _mk(ah, sg, T=400, seed=0):
    r = np.random.default_rng(seed)
    y = np.zeros(T)
    y[0] = r.normal(0, _true_sd)
    for t in range(1, T):
        y[t] = ah*y[t-1] + r.normal(0, sg)
    return y
def _batch(ah, sg, n=60, T=400, seed0=0):
    return [_mk(ah, sg, T=T, seed=seed0+s) for s in range(n)]

_REF = _batch(0.95, 1.0, n=200, seed0=900_000)          # 一批**真实**序列

print('  (a) 为什么必须在一批上做：单条序列的 s2/s1 抖动比信号还大')
_one = np.array([_mk(0.95, 1.0, seed=s)[200:].std() / _mk(0.95, 1.0, seed=s)[:200].std()
                 for s in range(200)])
_neff = (400/2)*(1-_a*_a)/(1+_a*_a)
print(f'    单条 s2/s1（T=400, 200 seeds）: 中位 {np.median(_one):.3f}, '
      f'p5 {np.percentile(_one,5):.3f}, p95 {np.percentile(_one,95):.3f}')
print(f'    理论有效样本量 n_eff = (T/2)(1−a²)/(1+a²) = {_neff:.1f}  ->  '
      f'sd(sd) ≈ 1/sqrt(2·n_eff) = {1/np.sqrt(2*_neff):.3f}')
assert np.percentile(_one, 95) - np.percentile(_one, 5) > 0.5
print('    -> p5–p95 跨了 '
      f'{np.percentile(_one,5):.2f}–{np.percentile(_one,95):.2f}，'
      '**任何单条容差都不可能工作**')

print()
print('  (b) 为什么能量比要相对**真实批**而不是理论值:')
_med_ref = float(np.median([y[200:].std() for y in _REF]))
print(f'    理论稳态 sd = {_true_sd:.4f}')
print(f'    真实批的 median(后半 sd) = {_med_ref:.4f}  ->  估计量本身只有理论值的 '
      f'{_med_ref/_true_sd:.4f}')
assert _med_ref / _true_sd < 0.95, 'sd 估计量的中位数应明显低于真值'
print('    原因：sd 估计量右偏，而 n_eff≈10 时中位数明显低于真值。')
print('    所以拿理论值做分母会把一个**好**模型误判成「偏静止」。')
_bad_er = float(np.median([y[200:].std() for y in _batch(0.95, 1.0)])) / _true_sd
_good_er = float(np.median([y[200:].std() for y in _batch(0.95, 1.0)])) / _med_ref
print(f'    同一批生成序列: 相对理论值 {_bad_er:.4f}（会误判）'
      f'  vs  相对真实批 {_good_er:.4f}（正确）')
assert _bad_er < 0.88 and abs(_good_er - 1.0) < 0.15

print()
print('  (c) 四种情形的诊断:')
print('    情形                     能量比      平稳比    平稳?    诊断')
for _tag, _ah, _sg, _expect in [
        ('â=0.95 σ=1.0（正确）', 0.95, 1.0, 'ok'),
        ('â=0.88（低估持续性）', 0.88, 1.0, 'fixed_point'),
        ('σ=0.5（低估噪声）', 0.95, 0.5, 'fixed_point'),
        ('σ=2.0（高估噪声）', 0.95, 2.0, 'fixed_point'),
        ('â=1.02（不稳定）', 1.02, 1.0, 'compounding')]:
    _d = diagnose_drift(_batch(_ah, _sg), _REF)
    print(f'    {_tag:24s} {_d["energy_ratio"]:9.3f}   {_d["stat_ratio"]:8.3f}   '
          f'{str(_d["is_stationary"]):6s}  {_d["verdict"]}')
    assert _d['verdict'] == _expect, f'{_tag}: 期望 {_expect}，得到 {_d["verdict"]}'

_dl = diagnose_drift(_batch(0.95, 0.5), _REF)
_dh = diagnose_drift(_batch(0.95, 2.0), _REF)
assert _dl['energy_ratio'] < 1 < _dh['energy_ratio']

print()
print('✅ 五种情形被正确区分')
print(f'✅ 两种「不动点错」的能量比一个 {_dl["energy_ratio"]:.3f}（偏静止）、'
      f'一个 {_dh["energy_ratio"]:.3f}（偏乱），但**都平稳**')
print('✅ 而不稳定的那一条不平稳 —— 这是区分两支的关键信号')
print()
print('   三个可操作的结论:')
print('     · 平稳性诊断必须在**批**上做：单条序列的抖动比信号大。')
print('     · 能量比必须相对**真实批**算：同一个统计量两边都算一次，估计量偏差自动抵消。')
print('     · 两支的修法相反 —— fixed_point 调采样参数，compounding 改训练目标/加锚帧。')
print('       而只看「后半段比前半段糊」这一个现象分不出是哪一支。')

## 📖 参考答案

In [ ]:
def fit_a_multistep(x, k, grid=None):
    '''最小化 k 步 rollout 总误差的 a。'''
    if grid is None:
        grid = np.linspace(-1.2, 1.2, 2401)
    n = len(x)
    best, best_loss = grid[0], np.inf
    for a in grid:
        loss = 0.0
        p = 1.0
        for j in range(1, k+1):
            p *= a
            if n - j <= 0:
                break
            d = p * x[:n-j] - x[j:]
            loss += float(d @ d)
        if loss < best_loss:
            best, best_loss = float(a), loss
    return best

def collapse_rate(a, n, seeds=400, horizon=200, thresh=10.0):
    '''崩坏率与未崩坏样本的方差中位数。'''
    bad, vars_ok = 0, []
    for s in range(seeds):
        x = ar1(a, n, 10_000 + s)
        aa = fit_a(x)
        sg = (x[1:] - aa * x[:-1]).std()
        y = np.zeros(horizon)
        y[0] = x[-1]
        rr = np.random.default_rng(60_000 + s)
        for t in range(1, horizon):
            y[t] = aa * y[t-1] + rr.normal(0, sg)
        if not np.isfinite(y).all() or np.abs(y).max() > thresh * x.std():
            bad += 1
        else:
            vars_ok.append(y[horizon//2:].var())
    med = float(np.median(vars_ok)) if vars_ok else float('nan')
    return bad / seeds, med

def anchor_error(ah, T, m):
    '''间隔 m 的锚帧下第 T 帧的误差放大倍数。'''
    last_anchor = (T // m) * m
    return float(ah ** (T - last_anchor))

def diagnose_drift(gen_seqs, ref_seqs, tol=0.20):
    '''区分「不动点错」与「误差还在累积」。两个统计量都在批上算。'''
    def halves(seqs):
        r, s2s = [], []
        for y in seqs:
            h = len(y) // 2
            s1, s2 = float(np.std(y[:h])), float(np.std(y[h:]))
            r.append(s2 / max(s1, 1e-12))
            s2s.append(s2)
        return float(np.median(r)), float(np.median(s2s))
    stat_ratio, med_gen = halves(gen_seqs)
    _, med_ref = halves(ref_seqs)
    energy_ratio = med_gen / max(med_ref, 1e-12)
    is_stat = abs(stat_ratio - 1.0) < tol
    if not is_stat:
        verdict = 'compounding'
    elif abs(energy_ratio - 1.0) < tol:
        verdict = 'ok'
    else:
        verdict = 'fixed_point'
    return {'energy_ratio': energy_ratio, 'stat_ratio': stat_ratio,
            'is_stationary': bool(is_stat), 'verdict': verdict}

print('参考答案已定义。')
print()
print('要点：')
print('  1. 多步损失**不**修一步拟合的向下偏差；它在模型正确指定时几乎什么也不改。')
print('     它的真实作用是在**误配**下把精度从短跨度挪到长跨度。')
print('  2. 「中位数 + 崩坏率」缺一不可：它们分别捕捉相反症状的两支。')
print('  3. 锚帧把误差从 |â|^T 变成 |â|^(<=m)，在不稳定动力学下是指数级改善。')
print('  4. 「不动点错」与「误差累积」的区分信号是**序列是否平稳**，')
print('     而它必须在**一批**上做（单条 s2/s1 在 T=400 时 p5–p95 跨 0.67–1.59），')
print('     且能量比要相对**真实批**算 —— sd 估计量的中位数本身只有真值的 0.885。')

## 🧪 真实工程胶囊：长视频生成的漂移体检

下面这段代码在一批生成的序列上跑完本模块的四项诊断，
并按「哪一支」给出**对应的修法**——而不是笼统地说「有漂移」。

关键设计：它先用**崩坏率**把两支分开，再对未崩坏的部分做平稳性诊断。
这个顺序是必要的：崩坏样本会让任何方差诊断失去意义（第 3 节的 $10^{14}$）。

In [ ]:
def drift_report(gen_seqs, ref_seqs, thresh=10.0, tol=0.20):
    '''一批生成序列的漂移体检。ref_seqs 是一批**真实**序列（同口径的参照）。'''
    train_sd = float(np.median([np.std(y) for y in ref_seqs]))
    n = len(gen_seqs)
    collapsed = [y for y in gen_seqs
                 if (not np.isfinite(y).all()) or np.abs(y).max() > thresh * train_sd]
    ok_seqs = [y for y in gen_seqs
               if np.isfinite(y).all() and np.abs(y).max() <= thresh * train_sd]
    rate = len(collapsed) / n
    print(f'  样本数 {n}，崩坏 {len(collapsed)}（{rate*100:.1f}%）')
    if not ok_seqs:
        print('  ⚠️  全部崩坏 —— 后续诊断无意义')
        return dict(collapse_rate=rate, verdict='all_collapsed', fixes=['降低 |â|（多步损失/正则）'])

    d = diagnose_drift(ok_seqs, ref_seqs, tol=tol)
    maj, er = d['verdict'], d['energy_ratio']
    print(f'  未崩坏样本（批级）: 能量比 {er:.3f}，平稳比 {d["stat_ratio"]:.3f}，'
          f'平稳 = {d["is_stationary"]}，判定 = {maj}')

    fixes = []
    if rate > 0.02:
        fixes.append(f'崩坏率 {rate*100:.1f}% > 2%：一部分模型的 |â|>1。'
                     f'修法 = 多步损失 / 谱范数约束 / 更长的训练序列')
    if maj == 'compounding':
        fixes.append('主体仍在累积（不平稳）：修法 = 多步/rollout 损失，或加锚帧'
                     '（把有效 rollout 长度压到 m）')
    elif maj == 'fixed_point':
        d = '偏静止/糊' if er < 1 else '偏乱/过锐'
        fixes.append(f'主体已平稳但不动点错（能量比 {er:.3f}，{d}）：'
                     f'修法 = 调采样参数（温度 / CFG 强度 / 噪声调度），**不是**改训练目标')
    elif maj == 'ok' and rate <= 0.02:
        fixes.append('无明显问题')
    print('  ' + '-' * 70)
    for f in fixes:
        print(f'  -> {f}')
    print('  ' + '-' * 70)
    print()
    return dict(collapse_rate=rate, energy_ratio=er, verdict=maj, fixes=fixes)

def _gen_batch(a_hat, noise_sd, n=60, T=400, x0_sd=None, seed0=0):
    '''模拟一批「生成」出来的序列。'''
    x0_sd = x0_sd if x0_sd is not None else 1.0/np.sqrt(max(1-0.95**2, 1e-9))
    out = []
    for s in range(n):
        r = np.random.default_rng(seed0 + s)
        y = np.zeros(T)
        y[0] = r.normal(0, x0_sd)
        for t in range(1, T):
            y[t] = a_hat * y[t-1] + r.normal(0, noise_sd)
        out.append(y)
    return out

TRUE_A, TRUE_SD = 0.95, 1.0/np.sqrt(1-0.95**2)
REF_BATCH = _gen_batch(TRUE_A, 1.0, n=200, seed0=900_000)     # 一批**真实**序列作参照
print(f'训练分布: a={TRUE_A}, 噪声 sd=1.0, 稳态 sd={TRUE_SD:.4f}')
print(f'参照批（200 条真实序列）的 median(全段 sd) = '
      f'{np.median([np.std(y) for y in REF_BATCH]):.4f}')
print()

print('=== 场景 A：模型很好 ===')
_A = drift_report(_gen_batch(0.95, 1.0), REF_BATCH)
assert _A['verdict'] == 'ok', f'应判为 ok，得到 {_A["verdict"]}'

print('=== 场景 B：â 被低估（生成变静止）===')
_B = drift_report(_gen_batch(0.88, 1.0), REF_BATCH)
assert _B['verdict'] == 'fixed_point' and _B['energy_ratio'] < 1, \
    f'应判为不动点错且偏静止，得到 {_B["verdict"]}, {_B["energy_ratio"]:.3f}'
assert any('采样参数' in f for f in _B['fixes'])

print('=== 场景 C：噪声尺度高估（生成变乱）===')
_C = drift_report(_gen_batch(0.95, 1.8), REF_BATCH)
assert _C['energy_ratio'] > 1, f'应偏乱，得到 {_C["energy_ratio"]:.3f}'

print('=== 场景 D：一部分模型不稳定（混合批）===')
_mix = _gen_batch(0.95, 1.0, n=50, seed0=0) + _gen_batch(1.02, 1.0, n=10, seed0=500)
_D = drift_report(_mix, REF_BATCH)
assert _D['collapse_rate'] > 0.1, f'应检出崩坏，得到 {_D["collapse_rate"]:.3f}'
assert any('崩坏率' in f for f in _D['fixes'])

print('=== 场景 E：全部不稳定 ===')
_E = drift_report(_gen_batch(1.03, 1.0), REF_BATCH)
assert _E['verdict'] == 'all_collapsed'

print('工程含义：')
print('  · 诊断顺序是**必要的**：先按崩坏率把两支分开，再对未崩坏部分做平稳性诊断。')
print('    如果先算方差，场景 D 的均值会被那 10 个崩坏样本主导（第 3 节的 1e14）。')
print('  · 场景 B 与 C 的现象都是「后半段看起来不对」，而修法相反 ——')
print('    B 要调大采样温度/降 CFG，C 要调小。')
print('    如果只看「有漂移」就去加多步损失，两个场景都修不好。')
print('  · 场景 A 的判定是 ok，说明这套诊断不会对好模型误报。')